This is one of a series of Colab notebooks created for the **IR course**. It demonstrates how we can index a collection, and how to access an index to highlight some index analysis.

The **learning outcomes** of the this notebook are:


*   PyTerrier setup.
*   Preprocessing.
*   Indexing a collection.
*   Accessing and exploring the index.

What is PyTerrier?

**[PyTerrier](https://pyterrier.readthedocs.io/en/latest/)** is a Python framework, but uses the underlying [Terrier information retrieval](http://terrier.org/) toolkit for many indexing and retrieval operations. Terrier is written in Java and has a long history dating back to 2001. PyTerrier makes it easy to perform IR experiments in Python, but using the mature Terrier platform for the expensive indexing and retrieval operations.

### **Setup**
We will first install Pyterrier as follows:

In [1]:
#install the Pyterrier framework
#!pip install python-terrier

The next step is to initialise PyTerrier. This is performed using PyTerrier's init() method. The init() method is needed as PyTerrier must download Terrier's jar file and start the Java virtual machine. We prevent init() from being called more than once by checking started().

In [ ]:
import pyterrier as pt

if not pt.started():
  pt.init()

C:\Users\Mazen\AppData\Local\Temp\ipykernel_3276\1742394333.py:2: DeprecationWarning: Call to deprecated function (or staticmethod) started. (use pt.java.started() instead) -- Deprecated since version 0.11.0.
  if not pt.started():
Java started and loaded: pyterrier.java.colab, pyterrier.java, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]
C:\Users\Mazen\AppData\Local\Temp\ipykernel_3276\1742394333.py:3: DeprecationWarning: Call to deprecated method pt.init(). Deprecated since version 0.11.0.
java is now started automatically with default settings. To force initialisation early, run:
pt.java.init() # optional, forces java initialisation
  pt.init()


We will import all the python libraries needed for this lab

In [3]:
#we need to import the following libraries.
import pandas as pd
import re
import os

import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
nltk.download('stopwords')
nltk.download('punkt_tab')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Mazen\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Mazen\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Mazen\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

### **Data preparation**
We will first create five textual documents.

In [ ]:
docs_df = pd.DataFrame([["d0", "breakthrough drug for schizophrenia"],
                        ["d1", "new schizophrenia drug"],
                        ["d2", "new approach for treatment of schizophrenia"],
                        ["d3", "new hopes for schizophrenia patients"] ],
                        columns=["docno", "raw_text"])

docs_df

,docno,raw_text
0,d0,breakthrough drug for schizophrenia
1,d1,new schizophrenia drug
2,d2,new approach for treatment of schizophrenia
3,d3,new hopes for schizophrenia patients


Before indexing our data we need to do the following processing steps:


1.   **Remove stopwords.**
3.   **Stemming.**



In [5]:
def preprocess_text(text):
    stemmer = PorterStemmer()
    stop_words = set(stopwords.words('english'))

    text = text.lower()
    tokens = word_tokenize(text)
    tokens = [token for token in tokens if token.isalnum() and token not in stop_words]
    tokens = [stemmer.stem(token) for token in tokens]
    result = ' '.join(tokens)

    return result

In [6]:
# Apply the preprocess function on each document
docs_df['processed_text'] = docs_df['raw_text'].apply(preprocess_text)
print(docs_df[['docno', 'raw_text', 'processed_text']])

  docno                                     raw_text  \
0    d0          breakthrough drug for schizophrenia   
1    d1                       new schizophrenia drug   
2    d2  new approach for treatment of schizophrenia   
3    d3         new hopes for schizophrenia patients   

                         processed_text  
0       breakthrough drug schizophrenia  
1                new schizophrenia drug  
2  new approach treatment schizophrenia  
3        new hope schizophrenia patient  


Next, we will index the dataframe's documents. The index, with all its data structures, is saved into a directory called **myFirstIndex**.

In [ ]:
current_dir = os.getcwd()
index_path = os.path.join(current_dir, "invertedIndex")

# Create an indexer for English text
indexer = pt.DFIndexer(index_path, overwrite=True)

# Index the documents; assuming `docs_df` has columns `processed_text` (text) and `docno` (document IDs)
index_ref = indexer.index(docs_df["processed_text"], docs_df["docno"])

# Output index information
print('========================================================================================')
print(index_ref.toString())

c:\Users\Mazen\Desktop\AAST\Feb 2026\Information Retrieval\week 5\invertedIndex/data.properties


### **Explore the index**
An index has several data structures:

*    **the CollectionStatistics**- the salient global statistics of the index. Contains metrics such as the total number of documents, total number of terms, and overall term frequencies.
*    **the Lexicon** - the vocabulary of the index, including statistics of the terms, and a pointer into the inverted index. It serves as the vocabulary of the index. It contains unique terms extracted from the collection, stores statistics for each term (e.g., document frequency, collection frequency) and maintains pointers (or offsets) to the corresponding posting list in the inverted index.

* **the inverted index (a PostingIndex**) - contains the posting list for each term, detailing the frequency in which aterm appears in that document. It contains posting lists for each term. A posting list is essentially a list of documents in which the term appears. Each entry in a posting list typically includes the document identifier and the frequency (or positions) of the term in that document.
* **the DocumentIndex** - Stores metadata such as the length of each document and field-specific lengths if the document has multiple fields (e.g., title, body).
* **the MetaIndex** - contains document metadata, such as the docno, and optionally the raw text and the URL of each document.
* **the direct index (also a PostingIndex)** - contains a posting list for each document, detailing which terms occuring in that document and which frequency. The presence of the direct index depends on the Indexing Type that has been applied - single-pass and some memory indices do not provide a direct index.


Let's check the files the index files created.

We can export our index into our machine as follows:

Let's check the statistics about the index we created.

In [9]:
print(index_ref.toString())

#we will first load the index
index = pt.IndexFactory.of(index_ref)

#we will call getCollectionStatistics() to check the stats
print(index.getCollectionStatistics())

c:\Users\Mazen\Desktop\AAST\Feb 2026\Information Retrieval\week 5\invertedIndex/data.properties
Number of documents: 4
Number of terms: 8
Number of postings: 14
Number of fields: 0
Number of tokens: 14
Field names: []
Positions:   false



We can check the lexicon which is the **vocabulary** of the collection.

* Nt is the number of unique documents that each term occurs in.
* TF is the total number of occurrences – some weighting models use this instead of Nt.
* The numbers in the @{} are a pointer – they tell Terrier where the postings are for that term in the inverted index data structure.


In [10]:
for kv in index.getLexicon():
  print("%s -> %s " % (kv.getKey(), kv.getValue().toString()))

approach -> term5 Nt=1 TF=1 maxTF=1 @{0 0 0} 
breakthrough -> term1 Nt=1 TF=1 maxTF=1 @{0 0 4} 
drug -> term0 Nt=2 TF=2 maxTF=1 @{0 0 6} 
hope -> term7 Nt=1 TF=1 maxTF=1 @{0 1 2} 
new -> term3 Nt=3 TF=3 maxTF=1 @{0 2 0} 
patient -> term6 Nt=1 TF=1 maxTF=1 @{0 3 0} 
schizophrenia -> term2 Nt=4 TF=4 maxTF=1 @{0 3 6} 
treatment -> term4 Nt=1 TF=1 maxTF=1 @{0 4 6} 


we can also lookup a term in PyTerrier's lexicon:

In [11]:
index.getLexicon()["schizophrenia"].toString()

'term2 Nt=4 TF=4 maxTF=1 @{0 3 6}'

In [12]:
index.getLexicon()["schizophrenia"]

<org.terrier.structures.LexiconEntry at 0x1aee53c41d0 jclass=org/terrier/structures/LexiconEntry jself=<LocalRef obj=0x7a0d7302 at 0x1aee53c6eb0>>

**The inverted index** tells us in which documents each term occurs in.
The LexiconEntry is the pointer that tell us where to find the postings for that term in the inverted index.

Let's look in which documents the word "schizophrenia" occurs and its frequency in each document.

**Note:** we need to preprocess each search term with the same preprocessing steps we performed on the collection.

In [ ]:
term="schizophrenia"

try:
 pointer = index.getLexicon()[term]
 for posting in index.getInvertedIndex().getPostings(pointer):
    print(posting.toString() + " doclen=%d" % posting.getDocumentLength())
except:
    print("term %s not found"%term)

ID(0) TF(1) doclen=3
ID(1) TF(1) doclen=3
ID(2) TF(1) doclen=4
ID(3) TF(1) doclen=4


How many documents does term "schizophrenia" occur in?

In [14]:
index.getLexicon()[term].getDocumentFrequency()

4

What terms occur in the 4th document?

In [15]:
di = index.getDirectIndex()
doi = index.getDocumentIndex()
lex = index.getLexicon()
docid = 2 #docids are 0-based #note: postings will be null if the document is empty
for posting in di.getPostings(doi.getDocumentEntry(docid)):
    termid = posting.getId()
    lee = lex.getLexiconEntry(termid)
    print("%s with frequency %d" % (lee.getKey(),posting.getFrequency()))

schizophrenia with frequency 1
new with frequency 1
treatment with frequency 1
approach with frequency 1


In [ ]:
# Define a function to retrieve document IDs for a given term
def get_document_ids_for_term(index, term):
    try:
        pointer = index.getLexicon()[term]
        doc_ids = [posting.getId() for posting in index.getInvertedIndex().getPostings(pointer)]
        return doc_ids  # Return as a list for easier set operations
    except Exception:
        print("Term '%s' not found" % term)
        return []  # Return an empty list if the term is not found

# Function to merge two posting lists
def merge(p1, p2):
    answer = []
    i, j = 0, 0
    while i < len(p1) and j < len(p2):
        if p1[i] == p2[j]:  # If both doc IDs are the same
            answer.append(p1[i])
            i += 1
            j += 1
        elif p1[i] < p2[j]:  # Move pointer in the smaller list
            i += 1
        else:
            j += 1
    return answer

In [18]:
# Define your query
query = "drug AND schizophrenia"
terms = query.split(' AND ')

# Retrieve document IDs for each term
postings_lists = {term: get_document_ids_for_term(index, term) for term in terms}

# Check the terms and calculate the intersection
if len(terms) < 2:
    print("Not enough terms to perform a merge.")
else:
    # Loop through terms to find the intersection
    result = postings_lists[terms[0]]  # Start with the first term's postings
    for i in range(1, len(terms)):
        operand = postings_lists[terms[i]]
        result = merge(result, operand)  # Update the result with the merging

    print("Documents matching the AND query:", result)

Documents matching the AND query: ['0', '1']
